# Machine Learning Interview Preparation

Practical examples use scikit-learn and synthetic datasets. Install dependencies if needed: `pip install numpy pandas scikit-learn matplotlib`.

## 1. Create a reproducible synthetic classification dataset

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

X, y = make_classification(
    n_samples=1200,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    weights=[0.88, 0.12],
    random_state=42,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())


## 2. Establish a baseline

A baseline helps verify that a more complex model adds value.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score, average_precision_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
base_prob = baseline.predict_proba(X_test)[:, 1]

print("Baseline accuracy:", accuracy_score(y_test, base_pred))
print("Baseline positive recall:", recall_score(y_test, base_pred, zero_division=0))
print("Baseline PR-AUC:", average_precision_score(y_test, base_prob))


## 3. Train and evaluate a model

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

classifier = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
classifier.fit(X_train, y_train)
pred = classifier.predict(X_test)
prob = classifier.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))
print("ROC-AUC:", roc_auc_score(y_test, prob))
print("PR-AUC:", average_precision_score(y_test, prob))


## 4. Tune a decision threshold on validation data

Do not choose the threshold on the final test set. Here, a separate validation split is created from the training data.

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

X_fit, X_valid, y_fit, y_valid = train_test_split(
    X_train, y_train, test_size=0.25, stratify=y_train, random_state=7
)
threshold_model = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
threshold_model.fit(X_fit, y_fit)
valid_prob = threshold_model.predict_proba(X_valid)[:, 1]

candidates = []
for threshold in np.linspace(0.05, 0.95, 91):
    valid_pred = (valid_prob >= threshold).astype(int)
    precision = precision_score(y_valid, valid_pred, zero_division=0)
    recall = recall_score(y_valid, valid_pred, zero_division=0)
    if recall >= 0.90:
        candidates.append((threshold, precision, recall, valid_pred.mean()))

choice = max(candidates, key=lambda row: row[0]) if candidates else None
print("threshold, precision, recall, predicted-positive-rate:", choice)


## 5. Cross-validation without preprocessing leakage

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(
    classifier, X_train, y_train, cv=cv, scoring="average_precision"
)
print("Fold PR-AUC scores:", scores)
print("Mean PR-AUC:", scores.mean())


## 6. Regression baseline and metrics

In [ ]:
from sklearn.datasets import make_regression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from math import sqrt

Xr, yr = make_regression(n_samples=800, n_features=8, noise=20, random_state=42)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    Xr, yr, test_size=0.2, random_state=42
)

for name, regressor in [
    ("median baseline", DummyRegressor(strategy="median")),
    ("random forest", RandomForestRegressor(n_estimators=150, random_state=42)),
]:
    regressor.fit(Xr_train, yr_train)
    prediction = regressor.predict(Xr_test)
    print(name)
    print(" MAE:", mean_absolute_error(yr_test, prediction))
    print(" RMSE:", sqrt(mean_squared_error(yr_test, prediction)))
    print(" R2:", r2_score(yr_test, prediction))


## 7. Permutation importance

Permutation importance estimates how much a chosen score degrades when a feature is shuffled. Correlated features can make individual importances hard to interpret.

In [ ]:
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    classifier, X_test, y_test,
    scoring="average_precision", n_repeats=8, random_state=42
)
order = np.argsort(importance.importances_mean)[::-1]
for idx in order[:8]:
    print(idx, round(importance.importances_mean[idx], 4))


## 8. Error analysis

In [ ]:
from collections import Counter

errors = Counter()
for actual, predicted in zip(y_test, pred):
    if actual == 1 and predicted == 0:
        errors["false_negative"] += 1
    elif actual == 0 and predicted == 1:
        errors["false_positive"] += 1
    elif actual == 1 and predicted == 1:
        errors["true_positive"] += 1
    else:
        errors["true_negative"] += 1
print(errors)


## Interview prompts

1. Why can a random split be invalid for time-series or grouped data?
2. Why must preprocessing be inside a pipeline during cross-validation?
3. When is PR-AUC more informative than ROC-AUC?
4. How would you monitor a model when labels arrive weeks later?
5. What evidence would justify retraining or rolling back a production model?